# Tugas Mandiri Data Mining – Pertemuan 2
## Data Quality Audit dan Preprocessing Pipeline (Telco Customer Churn)

Notebook ini mengikuti urutan langkah pada laporan. Jalankan sel dari atas ke bawah. Angka yang muncul di output
adalah angka yang dipakai untuk mengisi laporan, terutama tabel eksperimen B.8.

**Alur kerja:** Persiapan → Muat data → Audit kualitas data → Perbaikan tipe data → Pisahkan X dan y → Split stratified →
Bangun pipeline preprocessing → Cek hasil transformasi → Eksperimen scaling → Eksperimen imbalance → Cross-validation → Demo kebocoran data → Ringkasan.

## Langkah 0 – Persiapan library

In [ ]:
# Jalankan baris di bawah sekali saja jika library belum terpasang (hapus tanda #):
# !pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

## Langkah 1 – Muat dataset
Letakkan file `Telco-Customer-Churn.csv` satu folder dengan notebook ini, atau ubah `DATA_PATH` sesuai lokasi file Anda.

In [ ]:
DATA_PATH = "Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)
print("Ukuran data (baris, kolom):", df.shape)
df.head()

## Langkah 2 – Audit kualitas data (jawaban B.1 dan B.2)
Tujuannya melihat kondisi data apa adanya sebelum diubah: tipe data, nilai kosong (termasuk yang "tersembunyi" berupa spasi),
duplikat, dan sebaran kategori.

In [ ]:
print("=== Tipe data tiap kolom ===")
print(df.dtypes)

print("\n=== Missing value (NaN asli) ===")
na = df.isna().sum()
print(na[na > 0] if (na > 0).any() else "Tidak ada NaN asli")

# Nilai kosong tersembunyi: sel yang hanya berisi spasi
blank = {c: int((df[c].astype(str).str.strip() == "").sum()) for c in df.columns}
print("\n=== Sel kosong tersembunyi (hanya spasi) ===")
print({k: v for k, v in blank.items() if v > 0} or "Tidak ada")

print("\ncustomerID unik semua?", df["customerID"].is_unique)
print("Baris duplikat (setelah customerID dibuang):", df.drop(columns=["customerID"]).duplicated().sum())

In [ ]:
# Jumlah kategori unik dan contoh nilainya (membantu menentukan nominal vs ordinal)
for c in df.select_dtypes(include="object").columns:
    if c in ("customerID", "TotalCharges"):
        continue
    print(f"{c:18s} ({df[c].nunique()}):", sorted(df[c].unique()))

## Langkah 3 – Perbaiki tipe data `TotalCharges`
Kolom ini terbaca sebagai teks karena ada sel berisi spasi. Konversi ke angka dengan `errors="coerce"`,
lalu periksa baris yang menjadi NaN.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges:", df["TotalCharges"].isna().sum(),
      f"({df['TotalCharges'].isna().mean()*100:.2f}%)")
print("\nBaris yang kosong:")
print(df.loc[df["TotalCharges"].isna(), ["tenure", "MonthlyCharges", "TotalCharges"]])
print("\nSemua baris kosong itu punya tenure = 0?",
      (df.loc[df["TotalCharges"].isna(), "tenure"] == 0).all())

## Langkah 4 – Profil numerik dan distribusi target (jawaban B.1 dan B.5)

In [ ]:
print(df[["tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2))

print("\n=== Distribusi target Churn ===")
counts = df["Churn"].value_counts()
print(counts)
print((df["Churn"].value_counts(normalize=True) * 100).round(2))
print(f"Rasio kelas mayoritas : minoritas = {counts.max() / counts.min():.2f} : 1")

In [ ]:
# Visualisasi singkat untuk lampiran laporan
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(data=df, x="Churn", ax=ax[0])
ax[0].set_title("Distribusi target Churn")

sns.countplot(data=df, x="Contract", hue="Churn", ax=ax[1])
ax[1].set_title("Churn berdasarkan jenis kontrak")
ax[1].tick_params(axis="x", rotation=15)

sns.boxplot(data=df, x="Churn", y="tenure", ax=ax[2])
ax[2].set_title("Tenure berdasarkan Churn")

plt.tight_layout()
plt.savefig("eda_churn.png", dpi=150)
plt.show()

## Langkah 5 – Pisahkan fitur (X) dan target (y), buang identitas
`customerID` dibuang karena hanya identitas. Target dikodekan `No → 0` dan `Yes → 1`.

In [ ]:
df_model = df.drop(columns=["customerID"])
y = df_model["Churn"].map({"No": 0, "Yes": 1})
X = df_model.drop(columns=["Churn"])
print("X:", X.shape, "| y:", y.shape)

## Langkah 6 – Split data latih dan uji secara stratified
Split dilakukan **sebelum** imputasi, scaling, encoding, atau SMOTE. Ini aturan utama agar tidak terjadi kebocoran data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Latih:", X_train.shape, "| Uji:", X_test.shape)
print("Proporsi churn – seluruh data:", round(y.mean(), 4),
      "| latih:", round(y_train.mean(), 4),
      "| uji:", round(y_test.mean(), 4))

## Langkah 7 – Bangun preprocessing pipeline (jawaban B.3 dan B.4)
- **Numerik kontinu** (`tenure`, `MonthlyCharges`, `TotalCharges`): imputasi median → StandardScaler
- **Ordinal** (`Contract`): OrdinalEncoder dengan urutan Month-to-month < One year < Two year
- **Nominal** (kolom kategorikal lain): OneHotEncoder
- **Biner** (`SeniorCitizen`): dibiarkan apa adanya (sudah 0/1)

Fungsi `make_preprocessor(scale=...)` dibuat agar scaling bisa dimatikan untuk eksperimen A.

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
ord_cols = ["Contract"]
bin_cols = ["SeniorCitizen"]
nom_cols = [c for c in X.columns if c not in num_cols + ord_cols + bin_cols]

print("Nominal :", nom_cols)

def make_preprocessor(scale=True):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    num_pipe = Pipeline(num_steps)

    ord_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]])),
    ])
    nom_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("ord", ord_pipe, ord_cols),
        ("nom", nom_pipe, nom_cols),
        ("bin", "passthrough", bin_cols),
    ])

## Langkah 8 – Cek hasil transformasi (jawaban B.10)
`fit_transform` hanya pada data latih; data uji cukup `transform`. Setelah itu pastikan tidak ada NaN,
semua fitur numerik, dan fitur kontinu sudah berskala standar.

In [ ]:
pre = make_preprocessor(scale=True)
Xtr_t = pre.fit_transform(X_train)   # belajar parameter dari data latih saja
Xte_t = pre.transform(X_test)        # data uji hanya ditransformasi

feat_names = pre.get_feature_names_out()
Xtr_df = pd.DataFrame(Xtr_t, columns=feat_names)

print("Bentuk data latih setelah transform:", Xtr_t.shape)
print("Bentuk data uji setelah transform  :", Xte_t.shape)
print("NaN pada data latih :", int(np.isnan(Xtr_t).sum()))
print("NaN pada data uji   :", int(np.isnan(Xte_t).sum()))
print("Semua kolom numerik :", Xtr_df.dtypes.nunique() == 1 and str(Xtr_df.dtypes.iloc[0]).startswith("float"))
print("\nRata-rata & std fitur kontinu (data latih, harus ≈ 0 dan ≈ 1):")
print(Xtr_df[["num__tenure", "num__MonthlyCharges", "num__TotalCharges"]].agg(["mean", "std"]).round(3))
print("\nDistribusi kelas data latih:", y_train.value_counts().to_dict())
print("Distribusi kelas data uji  :", y_test.value_counts().to_dict(), "(tidak di-resample)")

## Langkah 9 – Eksperimen A dan B: tanpa scaling vs StandardScaler (jawaban B.4)
KNN dan Logistic Regression dipakai hanya sebagai alat ukur dampak preprocessing, bukan model akhir.
Untuk Logistic Regression dicatat juga jumlah iterasi sampai konvergen (`n_iter`).

In [ ]:
def evaluate(model, X_te, y_te):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        "accuracy": accuracy_score(y_te, pred),
        "precision_churn": precision_score(y_te, pred, zero_division=0),
        "recall_churn": recall_score(y_te, pred),
        "f1_churn": f1_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
    }

hasil = []   # semua hasil eksperimen dikumpulkan di sini

models = {
    "KNN (k=15)": lambda: KNeighborsClassifier(n_neighbors=15),
    "LogReg":     lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
}

for kode, kondisi, scale in [("A", "Tanpa scaling", False), ("B", "Dengan StandardScaler", True)]:
    for nama, buat in models.items():
        pipe = Pipeline([("pre", make_preprocessor(scale)), ("m", buat())])
        t0 = time.perf_counter()
        pipe.fit(X_train, y_train)
        durasi = time.perf_counter() - t0
        r = evaluate(pipe, X_test, y_test)
        r.update({"kode": kode, "kondisi": kondisi, "model": nama, "waktu_fit_detik": durasi})
        if nama == "LogReg":
            r["n_iter"] = int(pipe.named_steps["m"].n_iter_[0])
        hasil.append(r)

tabel_ab = pd.DataFrame([h for h in hasil if h["kode"] in ("A", "B")])
cols = ["kode", "kondisi", "model", "accuracy", "precision_churn", "recall_churn",
        "f1_churn", "roc_auc", "waktu_fit_detik", "n_iter"]
tabel_ab[cols].round(4)

## Langkah 10 – Eksperimen C dan D: penanganan class imbalance (jawaban B.5 dan B.6)
Semua memakai Logistic Regression dengan preprocessing yang sama (ber-scaling):
- **C**: tanpa penanganan (baseline)
- **D1**: `class_weight="balanced"`
- **D2**: SMOTE di dalam `ImbPipeline` (hanya menyentuh data latih)

In [ ]:
# Distribusi kelas data latih sebelum dan sesudah SMOTE
Xtr_scaled = make_preprocessor(True).fit_transform(X_train)
Xs, ys = SMOTE(random_state=RANDOM_STATE).fit_resample(Xtr_scaled, y_train)
print("Data latih sebelum SMOTE:", y_train.value_counts().to_dict())
print("Data latih sesudah SMOTE:", pd.Series(ys).value_counts().to_dict())
print("Data uji (tidak diubah) :", y_test.value_counts().to_dict())

In [ ]:
skenario = [
    ("C",  "Tanpa penanganan imbalance",
     Pipeline([("pre", make_preprocessor(True)),
               ("m", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])),
    ("D1", "class_weight='balanced'",
     Pipeline([("pre", make_preprocessor(True)),
               ("m", LogisticRegression(class_weight="balanced", max_iter=5000,
                                        random_state=RANDOM_STATE))])),
    ("D2", "SMOTE (di dalam pipeline)",
     ImbPipeline([("pre", make_preprocessor(True)),
                  ("smote", SMOTE(random_state=RANDOM_STATE)),
                  ("m", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])),
]

for kode, kondisi, pipe in skenario:
    pipe.fit(X_train, y_train)
    r = evaluate(pipe, X_test, y_test)
    r.update({"kode": kode, "kondisi": kondisi, "model": "LogReg"})
    hasil.append(r)
    print(f"[{kode}] {kondisi}")
    print(confusion_matrix(y_test, pipe.predict(X_test)))
    print()

tabel_cd = pd.DataFrame([h for h in hasil if h["kode"] in ("C", "D1", "D2")])
tabel_cd[["kode", "kondisi", "accuracy", "precision_churn", "recall_churn", "f1_churn", "roc_auc"]].round(4)

## Langkah 11 – Cross-validation yang aman dari kebocoran (jawaban B.7 poin 3)
Karena preprocessing dan SMOTE berada di dalam pipeline, keduanya dihitung ulang pada setiap fold latih saja.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pipe_cv = ImbPipeline([
    ("pre", make_preprocessor(True)),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("m", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])
cv = cross_validate(pipe_cv, X_train, y_train, cv=skf,
                    scoring=["recall", "precision", "f1", "roc_auc"])
pd.DataFrame({k.replace("test_", ""): [v.mean(), v.std()]
              for k, v in cv.items() if k.startswith("test_")},
             index=["rata-rata", "std"]).round(4)

## Langkah 12 – Demonstrasi kebocoran data (pelengkap B.6 dan B.7)
Dua cara dibandingkan dengan model KNN (peka terhadap data sintetis):
- **Salah**: scaling dan SMOTE dijalankan pada seluruh data, baru dibagi.
- **Benar**: split dulu, semua preprocessing dan SMOTE hanya di data latih (lewat pipeline).

Skor cara yang salah biasanya lebih tinggi. Itu bukan karena modelnya lebih baik, tetapi karena informasi data uji ikut masuk.

In [ ]:
# --- Cara SALAH ---
X_all_t = make_preprocessor(True).fit_transform(X)                   # fit pada seluruh data
Xs_all, ys_all = SMOTE(random_state=RANDOM_STATE).fit_resample(X_all_t, y)  # SMOTE sebelum split
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(
    Xs_all, ys_all, test_size=0.2, stratify=ys_all, random_state=RANDOM_STATE)
knn_w = KNeighborsClassifier(n_neighbors=5).fit(Xtr_w, ytr_w)
salah = evaluate(knn_w, Xte_w, yte_w)

# --- Cara BENAR ---
knn_b = ImbPipeline([("pre", make_preprocessor(True)),
                     ("smote", SMOTE(random_state=RANDOM_STATE)),
                     ("m", KNeighborsClassifier(n_neighbors=5))]).fit(X_train, y_train)
benar = evaluate(knn_b, X_test, y_test)

pd.DataFrame({"Cara salah (bocor)": salah, "Cara benar (pipeline)": benar}).round(4)

## Langkah 13 – Ringkasan hasil dan penyimpanan
Tabel di bawah dipakai untuk mengisi kolom "Hasil" pada tabel B.8 di laporan.

In [ ]:
ringkas = pd.DataFrame(hasil)
urut = ["kode", "kondisi", "model", "accuracy", "precision_churn", "recall_churn",
        "f1_churn", "roc_auc", "waktu_fit_detik", "n_iter"]
ringkas = ringkas[[c for c in urut if c in ringkas.columns]].round(4)
ringkas.to_csv("hasil_eksperimen_B8.csv", index=False)
ringkas

In [ ]:
# (Opsional) simpan dataset akhir hasil preprocessing untuk lampiran
Xtr_final = pd.DataFrame(Xtr_t, columns=feat_names).assign(Churn=y_train.values)
Xte_final = pd.DataFrame(Xte_t, columns=feat_names).assign(Churn=y_test.values)
Xtr_final.to_csv("telco_train_preprocessed.csv", index=False)
Xte_final.to_csv("telco_test_preprocessed.csv", index=False)
print("Tersimpan:", Xtr_final.shape, Xte_final.shape)